# Hospital Financial Distress Prediction
This notebook documents the process of building a model to predict financial status for acute care facilities using 5 years of CMS Medicare Cost Reports (2019-2023)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, shap, warnings

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Data Loading 
Five years of CMS Medicare Cost Reports (2019-2023) are stacked into a single dataframe with a year column added for context.

In [2]:
df = pd.concat([
    pd.read_csv(f'data/CostReport_{year}_Final.csv').assign(Year=year)
    for year in range(2019, 2024)
], ignore_index=True)

print(f'Dataset Shape: {df.shape}')
df.head()

Dataset Shape: (30400, 118)


,rpt_rec_num,Provider CCN,Hospital Name,Street Address,City,State Code,Zip Code,County,Medicare CBSA Number,Rural Versus Urban,CCN Facility Type,Provider Type,Type of Control,Fiscal Year Begin Date,Fiscal Year End Date,FTE - Employees on Payroll,Number of Interns and Residents (FTE),Total Days Title V,Total Days Title XVIII,Total Days Title XIX,Total Days (V + XVIII + XIX + Unknown),Number of Beds,Total Bed Days Available,Total Discharges Title V,Total Discharges Title XVIII,Total Discharges Title XIX,Total Discharges (V + XVIII + XIX + Unknown),Number of Beds + Total for all Subproviders,Hospital Total Days Title V For Adults & Peds,Hospital Total Days Title XVIII For Adults & Peds,Hospital Total Days Title XIX For Adults & Peds,Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds,Hospital Number of Beds For Adults & Peds,Hospital Total Bed Days Available For Adults & Peds,Hospital Total Discharges Title V For Adults & Peds,Hospital Total Discharges Title XVIII For Adults & Peds,Hospital Total Discharges Title XIX For Adults & Peds,Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds,Cost of Charity Care,Total Bad Debt Expense,Cost of Uncompensated Care,Total Unreimbursed and Uncompensated Care,Total Salaries From Worksheet A,Overhead Non-Salary Costs,Depreciation Cost,Total Costs,Inpatient Total Charges,Outpatient Total Charges,Combined Outpatient + Inpatient Total Charges,Wage-Related Costs (Core),Wage-Related Costs (RHC/FQHC),Total Salaries (adjusted),Contract Labor: Direct Patient Care,Wage Related Costs for Part - A Teaching Physicians,Wage Related Costs for Interns and Residents,Cash on Hand and in Banks,Temporary Investments,Notes Receivable,Accounts Receivable,Less: Allowances for Uncollectible Notes and Accounts Receivable,Inventory,Prepaid Expenses,Other Current Assets,Total Current Assets,Land,Land Improvements,Buildings,Leasehold Improvements,Fixed Equipment,Major Movable Equipment,Minor Equipment Depreciable,Health Information Technology Designated Assets,Total Fixed Assets,Investments,Other Assets,Total Other Assets,Total Assets,Accounts Payable,"Salaries, Wages, and Fees Payable",Payroll Taxes Payable,Notes and Loans Payable (Short Term),Deferred Income,Other Current Liabilities,Total Current Liabilities,Mortgage Payable,Notes Payable,Unsecured Loans,Other Long Term Liabilities,Total Long Term Liabilities,Total Liabilities,General Fund Balance,Total Fund Balances,Total Liabilities and Fund Balances,DRG Amounts Other Than Outlier Payments,DRG Amounts Before October 1,DRG Amounts After October 1,Outlier Payments For Discharges,Disproportionate Share Adjustment,Allowable DSH Percentage,Managed Care Simulated Payments,Total IME Payment,Inpatient Revenue,Outpatient Revenue,Total Patient Revenue,Less Contractual Allowance and Discounts on Patients' Accounts,Net Patient Revenue,Less Total Operating Expense,Net Income from Service to Patients,Total Other Income,Total Income,Total Other Expenses,Net Income,Cost To Charge Ratio,Net Revenue from Medicaid,Medicaid Charges,Net Revenue from Stand-Alone CHIP,Stand-Alone CHIP Charges,Year
0,650479,201302,LINCOLNHEALTH,6 ST. ANDREWS LANE,BOOTHBAY HARBOR,ME,04538-,LINCOLN,99920.0,R,CAH,1,2,10/01/2018,12/31/2018,362.19,NaN,NaN,820.0,126.0,1793.0,25.0,9125.0,NaN,187.0,33.0,382.0,67.0,NaN,690.0,80.0,1377.0,21.0,7665.0,NaN,187.0,33.0,382.0,259986.0,1047204.0,851792.0,851792.0,6124146.0,15117362.0,928865.0,18206588.0,9440440.0,21196720.0,30637160.0,NaN,NaN,NaN,NaN,NaN,NaN,4425475.0,NaN,591564.0,6985850.0,NaN,3294397.0,NaN,NaN,23717347.0,2480754.0,5779808.0,62333238.0,630254.0,8436081.0,22873561.0,NaN,NaN,46762120.0,12633553.0,3335454.0,15969007.0,86448474.0,1617780.0,2630412.0,NaN,NaN,17032.0,1971307.0,17186883.0,NaN,9448243.0,NaN,3303566.0,12751809.0,29938692.0,56509782.0,56509782.0,86448474.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10567431.0,23595634.0,34163065.0,12768843.0,21394222.0,21973982.0,-579760.0,61805.0,-517955.0,NaN,-517955.0,0.594265,1692896.0,2

## Exploratory Data Analysis

In [3]:
# Create lists of potential features to imporve output readability
facility_cols = [
    'Rural Versus Urban',
    'CCN Facility Type',
    'Provider Type',
    'Type of Control'
]

bed_cols = [
    'Number of Beds',
    'Number of Beds + Total for all Subproviders',
    'Hospital Number of Beds For Adults & Peds',
]

patient_cols = [
    'Total Days Title V',
    'Hospital Total Days Title V For Adults & Peds',
    'Total Days Title XVIII',
    'Hospital Total Days Title XVIII For Adults & Peds',
    'Total Days Title XIX',
    'Hospital Total Days Title XIX For Adults & Peds',
    'Total Days (V + XVIII + XIX + Unknown)',
    'Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds',
    'Total Bed Days Available',
    'Hospital Total Bed Days Available For Adults & Peds',
    'Total Discharges Title V',
    'Hospital Total Discharges Title V For Adults & Peds',
    'Total Discharges Title XVIII',
    'Hospital Total Discharges Title XVIII For Adults & Peds',
    'Total Discharges Title XIX',
    'Hospital Total Discharges Title XIX For Adults & Peds',
    'Total Discharges (V + XVIII + XIX + Unknown)',
    'Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds'    
]

financial_cols = [
    # Loss due to uncompensated care
    'Cost of Charity Care',
    'Total Bad Debt Expense',
    'Cost of Uncompensated Care',
    'Total Unreimbursed and Uncompensated Care',

    # Charges and Revenue options
    'Inpatient Total Charges',
    'Outpatient Total Charges',
    'Combined Outpatient + Inpatient Total Charges',
    'Inpatient Revenue',
    'Outpatient Revenue',
    'Total Patient Revenue',

    # Target
    'Net Income'
]

identifier_cols = [
    'Year',
    'Hospital Name',
    'City',
    'State Code',
]

df[identifier_cols].isnull().sum()

Year             0
Hospital Name    0
City             0
State Code       0
dtype: int64

#### Evaluation of Facility Information Data

In [4]:
for col in facility_cols:
    print(df[col].value_counts())
    print()

Rural Versus Urban
U    17142
R    12872
Name: count, dtype: int64

CCN Facility Type
STH      16331
CAH       6876
PH        3146
LTCH      1765
RH        1716
CH         466
RNMHC       58
ORD         41
TC           1
Name: count, dtype: int64

Provider Type
1     23099
4      3082
2      1771
5      1745
7       407
9       152
3        62
6        54
12       18
10        6
11        4
Name: count, dtype: int64

Type of Control
2     12200
4      7154
1      2991
9      1881
11     1710
5      1279
10     1176
6       551
8       415
13      388
12      320
7       220
3       115
Name: count, dtype: int64



In [5]:
# Checking for mislabeled facilities before removing non-acute care categories
pd.crosstab(df['Provider Type'], df['CCN Facility Type'])

CCN Facility Type,CAH,CH,LTCH,ORD,PH,RH,RNMHC,STH,TC
Provider Type,,,,,,,,,
1,6762,27,1,41,27,2,2,16236,1
2,0,9,1754,0,4,1,1,2,0
3,0,3,0,0,1,1,0,57,0
4,0,4,0,0,3075,0,1,2,0
5,0,13,4,0,12,1712,2,2,0
6,0,1,0,0,2,0,51,0,0
7,0,398,0,0,9,0,0,0,0
9,112,11,1,0,16,0,1,11,0
10,1,0,5,0,0,0,0,0,0


In [6]:
df[(df['Provider Type'] == 9) & (df['CCN Facility Type'].isin(['CAH', 'STH']))][['Hospital Name', 'State Code', 'Year', 'CCN Facility Type']].drop_duplicates('Hospital Name').head(20)

,Hospital Name,State Code,Year,CCN Facility Type
28,PONTOTOC HEALTH SERVICES,MS,2019,CAH
38,MARTHAS VINEYARD HOSPITAL,MA,2019,CAH
59,CASTANER HOSPITAL,PR,2019,STH
162,CARTHAGE AREA HOSPITAL,NY,2019,CAH
200,THI - HARMON MEDICAL HOSP,NV,2019,STH
226,THE HEIGHTS HOSPITA,TX,2019,STH
233,ERLANGER BLEDSOE HOSPITAL,TN,2019,CAH
270,HOT SPRINGS COUNTY MEMORIAL,WY,2019,CAH
271,NORTHERN COCHISE COMMUNITY HOSPITAL,AZ,2019,CAH
635,CATSKILL REGIONAL MEDICAL CENTER GRO,NY,2019,CAH


In [7]:
# Reduce dataset to only acute-care hospitals based on CCN Facility type
df = df[df['CCN Facility Type'].isin(['STH', 'CAH', 'CH'])]
df.shape

(23673, 118)

In [8]:
for col in facility_cols:
    print(df[col].value_counts(dropna=False))

Rural Versus Urban
R      12390
U      11095
NaN      188
Name: count, dtype: int64
CCN Facility Type
STH    16331
CAH     6876
CH       466
Name: count, dtype: int64
Provider Type
1     23025
7       398
9       134
3        60
12       18
5        15
2        11
4         6
11        4
10        1
6         1
Name: count, dtype: int64
Type of Control
2     11444
4      3485
1      2808
9      1809
11     1683
5       437
8       410
13      364
10      351
12      305
6       287
7       209
3        81
Name: count, dtype: int64


In [9]:
df[df['Rural Versus Urban'].isnull()]['CCN Facility Type'].value_counts()

CCN Facility Type
CH     146
STH     42
Name: count, dtype: int64

In [10]:
df[df['Type of Control'] == 7][['Hospital Name', 'State Code', 'CCN Facility Type']].drop_duplicates('Hospital Name').head(20)

,Hospital Name,State Code,CCN Facility Type
11,SIOUX SAN HOSPITAL,SD,STH
123,SHRINERS HOSPITAL FOR CHILDREN,TX,CH
151,SANTA FE INDIAN HOSPITAL,NM,STH
157,STANDING ROCK IHS HOSPITAL,ND,STH
714,CROW/NORTHERN CHEYENNE INDIAN HOSPIT,MT,CAH
721,CASS LAKE INDIAN HOSPITAL,MN,CAH
1893,SAMUEL SIMMONDS MEMORIAL HOSPITAL,AK,CAH
1937,WRANGELL MEDICAL CENTER,AK,CAH
2103,HU HU KAM MEMORIAL HOSPITAL,AZ,CAH
3055,KANAKANAK HOSPITAL,AK,CAH


In [11]:
df[(df['Rural Versus Urban'].isnull()) & (df['CCN Facility Type'] == 'STH')][['Hospital Name', 'State Code', 'Type of Control']].drop_duplicates('Hospital Name')

,Hospital Name,State Code,Type of Control
59,CASTANER HOSPITAL,PR,2
62,CARE REGIONAL MEDICAL CENTER,TX,2
153,ROCKEFELLER UNIVERSITY HOSPITAL,NY,3
200,THI - HARMON MEDICAL HOSP,NV,4
213,EDWIN SHAW REHAB,OH,8
226,THE HEIGHTS HOSPITA,TX,2
230,SAN JORGE CHILDRENS HOSPITAL INC.,PR,2
241,GROVE CREEK MEDICAL CENTER,ID,1
263,BURDETT CARE CENTER,NY,6
6136,PREMIER ER PLUS-TEMPLE LLC,TX,4


#### Evaluation of Facility Capacity Data

In [12]:
df[bed_cols].isnull().sum()

Number of Beds                                 245
Number of Beds + Total for all Subproviders    223
Hospital Number of Beds For Adults & Peds      258
dtype: int64

In [13]:
df[bed_cols].describe().round(2)

,Number of Beds,Number of Beds + Total for all Subproviders,Hospital Number of Beds For Adults & Peds
count,23428.00,23450.00,23415.00
mean,225.72,247.03,193.68
std,10470.11,10465.75,10429.72
min,1.00,1.00,1.00
25%,25.00,25.00,25.00
50%,69.00,92.00,60.00
75%,200.00,226.00,168.00
max,1594784.00,1594796.00,1594778.00


In [14]:
# At the time of this project published public information states New York Presbyterian Weill Cornell Medical Center is the largest hospital by bed count with 2,850 beds
df[df['Number of Beds'] > 3000][identifier_cols + bed_cols].sort_values('Hospital Name')

,Year,Hospital Name,City,State Code,Number of Beds,Number of Beds + Total for all Subproviders,Hospital Number of Beds For Adults & Peds
728,2019,BEAR LAKE MEMORIAL HOSPITAL,MONTPELIER,ID,52913.0,52981.0,52913.0
7582,2020,MERCY HOSPITAL & MEDICAL CENTER,CHICAGO,IL,144837.0,144892.0,132.0
4404,2019,OKLAHOMA HEART HOSPITAL,OKLAHOMA CITY,OK,28401.0,28401.0,28401.0
10337,2020,WAYNE HOSPITAL COMPANY,GREENVILLE,OH,1594784.0,1594796.0,1594778.0


In [15]:
bed_outlier_ccns = df[df['Number of Beds'] > 3000]['Provider CCN'].tolist()
df[df['Provider CCN'].isin(bed_outlier_ccns)][identifier_cols + bed_cols].sort_values(['Hospital Name', 'Year'])

,Year,Hospital Name,City,State Code,Number of Beds,Number of Beds + Total for all Subproviders,Hospital Number of Beds For Adults & Peds
728,2019,BEAR LAKE MEMORIAL HOSPITAL,MONTPELIER,ID,52913.0,52981.0,52913.0
6274,2020,BEAR LAKE MEMORIAL HOSPITAL,MONTPELIER,ID,21.0,89.0,21.0
12408,2021,BEAR LAKE MEMORIAL HOSPITAL,MONTPELIER,ID,21.0,89.0,21.0
18895,2022,BEAR LAKE MEMORIAL HOSPITAL,MONTPELIER,ID,21.0,89.0,21.0
25311,2023,BEAR LAKE MEMORIAL HOSPITAL,MONTPELIER,ID,21.0,91.0,21.0
15974,2021,INSIGHTS HOSPITAL AND MEDICAL CENTER,CHICAGO,IL,52.0,91.0,37.0
20040,2022,INSIGHTS HOSPITAL AND MEDICAL CENTER,CHICAGO,IL,52.0,91.0,37.0
26552,2023,INSIGHTS HOSPITAL AND MEDICAL CENTER,CHICAGO,IL,52.0,91.0,37.0
2895,2019,MERCY HOSPITAL & MEDICAL CENTER,CHICAGO,IL,203.0,258.0,168.0
7582,2020,MERCY HOSPITAL & MEDICAL CENTER,CHICAGO,IL,144837.0,144892.0,132.0


In [16]:
bed_diff = df['Number of Beds'] - df['Hospital Number of Beds For Adults & Peds']
print(bed_diff.describe().round(2))
print(f"\nRows where A&P beds < Total beds: {(bed_diff > 0).sum()}")
print(f"Rows where difference > 50 beds: {(bed_diff > 50).sum()}")
print(f"Rows where difference > 100 beds: {(bed_diff > 100).sum()}")
print(f"Rows where A&P beds > Total beds: {(df['Hospital Number of Beds For Adults & Peds'] > df['Number of Beds']).sum()}")
print(f"Rows where A&P beds equal Total beds: {(df['Hospital Number of Beds For Adults & Peds'] == df['Number of Beds']).sum()}")

count     23415.00
mean         32.12
std         946.80
min           0.00
25%           0.00
50%           8.00
75%          29.00
max      144705.00
dtype: float64

Rows where A&P beds < Total beds: 15172
Rows where difference > 50 beds: 3606
Rows where difference > 100 beds: 1514
Rows where A&P beds > Total beds: 0
Rows where A&P beds equal Total beds: 8243


#### Evaluation of Facility Payor Data

In [17]:
df[patient_cols].isnull().sum()

Total Days Title V                                                         23006
Hospital Total Days Title V For Adults & Peds                              23083
Total Days Title XVIII                                                       436
Hospital Total Days Title XVIII For Adults & Peds                            493
Total Days Title XIX                                                        2400
Hospital Total Days Title XIX For Adults & Peds                             2750
Total Days (V + XVIII + XIX + Unknown)                                       287
Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds            299
Total Bed Days Available                                                     245
Hospital Total Bed Days Available For Adults & Peds                          255
Total Discharges Title V                                                   23050
Hospital Total Discharges Title V For Adults & Peds                        23050
Total Discharges Title XVIII

In [18]:
df[patient_cols].describe().round(2)

,Total Days Title V,Hospital Total Days Title V For Adults & Peds,Total Days Title XVIII,Hospital Total Days Title XVIII For Adults & Peds,Total Days Title XIX,Hospital Total Days Title XIX For Adults & Peds,Total Days (V + XVIII + XIX + Unknown),Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds,Total Bed Days Available,Hospital Total Bed Days Available For Adults & Peds,Total Discharges Title V,Hospital Total Discharges Title V For Adults & Peds,Total Discharges Title XVIII,Hospital Total Discharges Title XVIII For Adults & Peds,Total Discharges Title XIX,Hospital Total Discharges Title XIX For Adults & Peds,Total Discharges (V + XVIII + XIX + Unknown),Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds
count,667.00,590.00,23237.00,23180.00,21273.00,20923.00,23386.00,23374.00,23428.00,23418.00,623.00,623.00,23271.00,23271.00,20891.00,20891.00,23381.00,23381.00
mean,2313.04,1349.53,8772.72,7306.24,3607.72,2499.13,33887.26,25869.93,53282.01,43966.30,439.95,439.95,1628.37,1628.37,627.23,627.23,6565.51,6565.51
std,5301.49,3284.06,13703.90,11798.49,8160.66,5822.81,55166.35,42063.13,70725.83,55699.96,694.46,694.46,2355.93,2355.93,1356.79,1356.79,9824.31,9824.31
min,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,26.00,26.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
25%,99.50,20.00,1007.00,522.00,113.00,69.00,2618.25,1388.25,9125.00,9125.00,14.00,14.00,157.00,157.00,26.00,26.00,448.00,448.00
50%,607.00,176.50,2792.00,2090.50,743.00,473.00,9949.00,7741.50,24455.00,21170.00,150.00,150.00,587.00,587.00,146.00,146.00,2380.00,2380.00
75%,2520.50,1629.50,11304.00,9706.25,3341.00,2224.00,44479.25,35053.25,72261.50,60590.00,585.50,585.50,2255.00,2255.00,606.50,606.50,9307.00,9307.00
max,67185.00,39906.00,208248.00,177055.00,139073.00,101131.00,911866.00,689552.00,1040250.00,844610.00,7148.00,7148.00,31284.00,31284.00,19799.00,19799.00,168398.00,168398.00


In [19]:
df[df['Total Days Title XIX'].isnull()]['CCN Facility Type'].value_counts()

CCN Facility Type
CAH    1158
STH    1089
CH      153
Name: count, dtype: int64

In [20]:
df[df['Hospital Total Days Title XIX For Adults & Peds'].isnull()]['CCN Facility Type'].value_counts()

CCN Facility Type
CAH    1418
STH    1179
CH      153
Name: count, dtype: int64

In [21]:
df[df['Total Days Title XIX'].isnull()].groupby(['Hospital Name'])['Year'].count().sort_values(ascending=False).head(20)

Hospital Name
SHRINERS HOSPITAL FOR CHILDREN         27
SHRINERS HOSPITALS FOR CHILDREN        14
THE SHRINERS HOSPITAL FOR CHILDREN      7
HEMPHILL COUNTY HOSPITAL                6
OUR LADY OF THE LAKE ASSUMPTION COM     6
PHYSICIANS CARE SURGICAL HOSPITAL       6
FALLS COMMUNITY HOSPITAL AND CLINIC     6
PETALUMA VALLEY HOSPITAL                6
SEDAN CITY HOSPITAL                     6
SHAMROCK GENERAL HOSPITAL               6
JEFFERSON COUNTY HOSPITAL               6
HIRAM W DAVIS MEDICAL CENTER - MS       5
HELEN HAYES HOSPITAL                    5
ADVANCED SURGICAL HOSPITAL              5
HILLSBORO COMMUNITY HOSPITAL            5
HEBREW HOME & HOSPITAL                  5
GUTTENBERG MUNICIPAL HOSPITAL           5
THE CHILDRENS HOME OF PITTSBURGH        5
THE CENTER FOR SPINAL SURGERY           5
WARD MEMORIAL HOSPITAL                  5
Name: Year, dtype: int64

In [22]:
df[df['Hospital Total Days Title XIX For Adults & Peds'].isnull()].groupby(['Hospital Name'])['Year'].count().sort_values(ascending=False).head(20)

Hospital Name
SHRINERS HOSPITAL FOR CHILDREN          27
SHRINERS HOSPITALS FOR CHILDREN         14
WASHINGTON COUNTY HOSPITAL               9
THE SHRINERS HOSPITAL FOR CHILDREN       7
JEFFERSON COUNTY HOSPITAL                7
SHAMROCK GENERAL HOSPITAL                6
SEDAN CITY HOSPITAL                      6
PETALUMA VALLEY HOSPITAL                 6
OUR LADY OF THE LAKE ASSUMPTION COM      6
HEMPHILL COUNTY HOSPITAL                 6
FALLS COMMUNITY HOSPITAL AND CLINIC      6
PHYSICIANS CARE SURGICAL HOSPITAL        6
WILMINGTON TREATMENT CENTER INC          5
GREENWOOD COUNTY HOSPITAL                5
ADVENTHEALTH DURAND                      5
GREELEY COUNTY HEALTH SERVICES           5
TREGO COUNTY LEMKE MEMORIAL HOSPITAL     5
ADVANCED SURGICAL HOSPITAL               5
WEST TN HEALTHCARE VOLUNTEER HOSPITA     5
GRISELL MEMORIAL HOSPITAL DIST #1        5
Name: Year, dtype: int64

In [23]:
(df['Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds'] / 
 df['Hospital Total Discharges Title V For Adults & Peds']).describe().round(2)

count       623.00
mean       5000.99
std       24316.68
min           3.00
25%          30.13
50%          59.00
75%         281.29
max      352197.00
dtype: float64

In [24]:
(df['Total Bed Days Available'] - df['Hospital Total Bed Days Available For Adults & Peds']).describe().round(2)

count     23418.00
mean       9324.74
std       17210.33
min           0.00
25%           0.00
50%        2920.00
75%       10248.00
max      216721.00
dtype: float64

In [25]:
print(f"Rows where A&P bed days > Total bed days: {(df['Hospital Total Bed Days Available For Adults & Peds'] > df['Total Bed Days Available']).sum()}")
print(f"Rows where A&P bed days equal Total bed days: {(df['Hospital Total Bed Days Available For Adults & Peds'] == df['Total Bed Days Available']).sum()}")

Rows where A&P bed days > Total bed days: 0
Rows where A&P bed days equal Total bed days: 8236


In [26]:
df['avg_los'] = (df['Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds'] / 
                     df['Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds'])

df[df['avg_los'] > 365][['Hospital Name', 'State Code', 'Year', 'CCN Facility Type', 'avg_los']].sort_values('avg_los', ascending=False).head(20)

,Hospital Name,State Code,Year,CCN Facility Type,avg_los
16692,TERENCE CARDINAL COOKE HEALTH CARE,NY,2021,STH,20564.000000
19250,TERENCE CARDINAL COOKE HEALTH CARE,NY,2022,STH,2917.857143
25887,TERENCE CARDINAL COOKE HEALTH CARE,NY,2023,STH,2552.375000
9577,TERENCE CARDINAL COOKE HEALTH CARE,NY,2020,STH,2282.888889
7230,FLORIDA STATE HOSPITAL,FL,2020,STH,615.000000


In [27]:
df[df['Hospital Name'].isin(['TERENCE CARDINAL COOKE HEALTH CARE', 'FLORIDA STATE HOSPITAL'])][
    ['Hospital Name', 'Year', 'CCN Facility Type', 'avg_los']
].sort_values(['Hospital Name', 'Year'])

,Hospital Name,Year,CCN Facility Type,avg_los
951,FLORIDA STATE HOSPITAL,2019,STH,NaN
7230,FLORIDA STATE HOSPITAL,2020,STH,615.000000
14382,FLORIDA STATE HOSPITAL,2021,STH,NaN
22854,FLORIDA STATE HOSPITAL,2022,STH,NaN
29634,FLORIDA STATE HOSPITAL,2023,STH,139.071429
9577,TERENCE CARDINAL COOKE HEALTH CARE,2020,STH,2282.888889
16692,TERENCE CARDINAL COOKE HEALTH CARE,2021,STH,20564.000000
19250,TERENCE CARDINAL COOKE HEALTH CARE,2022,STH,2917.857143
25887,TERENCE CARDINAL COOKE HEALTH CARE,2023,STH,2552.375000


In [28]:
df.drop(columns=['avg_los'], inplace=True)

#### Evaluation of Facility Financial Data

In [29]:
df[financial_cols].isnull().sum()

Cost of Charity Care                             2315
Total Bad Debt Expense                           1291
Cost of Uncompensated Care                       1133
Total Unreimbursed and Uncompensated Care        1022
Inpatient Total Charges                           232
Outpatient Total Charges                          234
Combined Outpatient + Inpatient Total Charges     192
Inpatient Revenue                                 593
Outpatient Revenue                                668
Total Patient Revenue                             567
Net Income                                        215
dtype: int64

In [30]:
df[financial_cols].describe().round(2)

,Cost of Charity Care,Total Bad Debt Expense,Cost of Uncompensated Care,Total Unreimbursed and Uncompensated Care,Inpatient Total Charges,Outpatient Total Charges,Combined Outpatient + Inpatient Total Charges,Inpatient Revenue,Outpatient Revenue,Total Patient Revenue,Net Income
count,2.135800e+04,2.238200e+04,2.254000e+04,2.265100e+04,2.344100e+04,2.343900e+04,2.348100e+04,2.308000e+04,2.300500e+04,2.310600e+04,2.345800e+04
mean,6.594032e+06,1.168243e+07,8.983498e+06,1.669125e+07,4.940515e+08,4.946739e+08,9.869989e+08,5.067338e+08,5.217526e+08,1.025636e+09,1.775548e+07
std,2.017270e+07,2.351603e+07,2.364482e+07,4.210347e+07,1.057672e+09,9.033970e+08,1.899212e+09,1.074664e+09,9.545758e+08,1.953690e+09,1.116010e+08
min,1.000000e+00,-9.501520e+05,-1.569430e+05,-6.502000e+03,1.000000e+00,7.000000e+01,1.000000e+00,1.000000e+00,-2.981246e+06,1.000000e+00,-3.955820e+09
25%,3.433172e+05,1.343864e+06,1.031000e+06,1.833624e+06,1.283332e+07,4.896458e+07,6.634720e+07,1.400115e+07,5.476362e+07,7.308925e+07,-1.202716e+06
50%,1.618331e+06,4.401657e+06,2.935379e+06,5.443640e+06,9.161651e+07,1.917036e+08,2.914430e+08,9.706486e+07,2.086481e+08,3.163288e+08,3.068474e+06
75%,5.714101e+06,1.289980e+07,8.415861e+06,1.644732e+07,5.283942e+08,6.065189e+08,1.152624e+09,5.426607e+08,6.373942e+08,1.207278e+09,1.747767e+07
max,8.192046e+08,6.715466e+08,8.504827e+08,1.063928e+09,2.447432e+10,2.337112e+10,4.511504e+10,2.461387e+10,2.337112e+10,4.511504e+10,7.919570e+09


In [31]:
df[df['Net Income'].isnull()][identifier_cols + facility_cols].value_counts(['CCN Facility Type'])

CCN Facility Type
CH                   146
STH                   68
CAH                    1
Name: count, dtype: int64

### Data Cleaning List
- Use CCN Based Filter to remove non-acute care facilities from the raw data frame due to more accurate reporting.
- Drop 188 rows with null Rural Versus Urban — small number & either special status or not located within one of the 50 US states.
- Group type of control into 3 groups: Non-Profit (1-2), For-Profit (3-6), & Government (7-13)
- Drop rows 728 (Bear Lake Memorial 2019), 7582 (Mercy Hospital 2020), 4404 (Oklahoma Heart 2019), 10337 (Wayne Hospital 2020) — confirmed single-year bed count reporting errors
- Use Hospital Number of Beds For Adults & Peds instead of Number of Beds
- Use Hospital Total Days For Adults & Peds columns for payer mix and utilization ratios verified excludes subcomponents in data dictionary
- Total Discharges and Hospital Total Discharges For Adults & Peds are functionally identical per the data dictionary
- Exclude Terence Cardinal Cooke Health Care — LTC facility & Florida State Hospital — state psychiatric institution
- Revisit nulls in both Net Income and Title XIX Discharges for A&P after removing items above. Currently 215 target nulls and 2750 Title XIX A&P.

- Use Total Unreimbursed and Uncompensated Care and Combined OP and IP Total Charges for Uncompensated care feature
- Use Hospital Total Bed Days Available for A&P not the basic metric.


In [32]:
data = df.copy()
print(f'Shape before exclusions: {data.shape}')

data = data.dropna(subset=['Rural Versus Urban'])
data = data.drop(index=[728, 7582, 4404, 10337])
data = data[~data['Hospital Name'].isin(['TERRENCE CARDINAL COOKE HEALTH CARE', 'FLORIDA STATE HOSPITAL'])]
print(f'Shape after exclusions: {data.shape}')

Shape before exclusions: (23673, 118)
Shape after exclusions: (23476, 118)


In [33]:
print(f'Null Net Income: {data['Net Income'].isnull().sum()}')
print(f'Null Title XIX Days: {data['Total Days Title XIX'].isnull().sum()}')
print(f'Null Title XIX Days for Adults & Peds: {data['Hospital Total Days Title XIX For Adults & Peds'].isnull().sum()}')

Null Net Income: 27
Null Title XIX Days: 2207
Null Title XIX Days for Adults & Peds: 2557


In [34]:
# Drop the 27 remaining facilities with a null target
data = data.dropna(subset=['Net Income'])
print(f'Shape after dropping null Net Income: {data.shape}')

Shape after dropping null Net Income: (23449, 118)


In [35]:
print(f"Unique facilities with null Title XIX A&P: {data[data['Hospital Total Days Title XIX For Adults & Peds'].isnull()]['Provider CCN'].nunique()}")
data[data['Hospital Total Days Title XIX For Adults & Peds'].isnull()].groupby(['CCN Facility Type'])['Provider CCN'].nunique()

Unique facilities with null Title XIX A&P: 889


CCN Facility Type
CAH    472
CH       4
STH    413
Name: Provider CCN, dtype: int64

In [36]:
pd.crosstab(
    data[data['Hospital Total Days Title XIX For Adults & Peds'].isnull()]['CCN Facility Type'],
    data[data['Hospital Total Days Title XIX For Adults & Peds'].isnull()]['Provider Type']
)

Provider Type,1,7,9,11,12
CCN Facility Type,,,,,
CAH,1393,0,25,0,0
CH,0,7,0,0,0
STH,1111,0,0,2,18


### Final Feature Null Analysis

In [37]:
medicare_nulls = ['Hospital Total Days Title XVIII For Adults & Peds', 'Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds']
medicaid_nulls = ['Hospital Total Days Title XIX For Adults & Peds', 'Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds']
bed_nulls = ['Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds', 'Hospital Total Bed Days Available For Adults & Peds']
los_nulls = ['Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds', 'Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds']
uncompensated_nulls = ['Total Unreimbursed and Uncompensated Care', 'Total Patient Revenue']
uncompensated_nulls2 = ['Total Unreimbursed and Uncompensated Care', 'Combined Outpatient + Inpatient Total Charges']

print(f'Medicare Nulls (medicaid_reported)')
print(f"Current rows: {len(data)}")
print(f"Rows with any null in required columns: {data[medicare_nulls].isnull().any(axis=1).sum()}")
print(f'Percent of current data retained if rows with any null in required columns dropped: {((len(data)-(data[medicare_nulls].isnull().any(axis=1).sum()))/len(data))}')
print()
print(data[medicare_nulls].isnull().sum().sort_values(ascending=False))

Medicare Nulls (medicaid_reported)
Current rows: 23449
Rows with any null in required columns: 300
Percent of current data retained if rows with any null in required columns dropped: 0.9872062774531963

Hospital Total Days Title XVIII For Adults & Peds                    300
Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds    111
dtype: int64


In [38]:
print(f'Medicaid Nulls (medicaid_reported)')
print(f"Current rows: {len(data)}")
print(f"Rows with any null in required columns: {data[medicaid_nulls].isnull().any(axis=1).sum()}")
print(f'Percent of current data retained if rows with any null in required columns dropped: {((len(data)-(data[medicaid_nulls].isnull().any(axis=1).sum()))/len(data))}')
print()
print(data[medicaid_nulls].isnull().sum().sort_values(ascending=False))

Medicaid Nulls (medicaid_reported)
Current rows: 23449
Rows with any null in required columns: 2556
Percent of current data retained if rows with any null in required columns dropped: 0.8909974839012325

Hospital Total Days Title XIX For Adults & Peds                      2556
Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds     111
dtype: int64


In [39]:
print(f'Bed Nulls (bed utilization reported)')
print(f"Current rows: {len(data)}")
print(f"Rows with any null in required columns: {data[bed_nulls].isnull().any(axis=1).sum()}")
print(f'Percent of current data retained if rows with any null in required columns dropped: {((len(data)-(data[bed_nulls].isnull().any(axis=1).sum()))/len(data))}')
print()
print(data[bed_nulls].isnull().sum().sort_values(ascending=False))

Bed Nulls (bed utilization reported)
Current rows: 23449
Rows with any null in required columns: 112
Percent of current data retained if rows with any null in required columns dropped: 0.99522367691586

Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds    111
Hospital Total Bed Days Available For Adults & Peds                   67
dtype: int64


In [40]:
print(f'LOS Nulls (length of stay reported)')
print(f"Current rows: {len(data)}")
print(f"Rows with any null in required columns: {data[los_nulls].isnull().any(axis=1).sum()}")
print(f'Percent of current data retained if rows with any null in required columns dropped: {((len(data)-(data[los_nulls].isnull().any(axis=1).sum()))/len(data))}')
print()
print(data[los_nulls].isnull().sum().sort_values(ascending=False))

LOS Nulls (length of stay reported)
Current rows: 23449
Rows with any null in required columns: 114
Percent of current data retained if rows with any null in required columns dropped: 0.9951383854322146

Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds          111
Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds    101
dtype: int64


In [41]:
print(f'Uncompensated Nulls (total patient revenue)')
print(f"Current rows: {len(data)}")
print(f"Rows with any null in required columns: {data[uncompensated_nulls].isnull().any(axis=1).sum()}")
print(f'Percent of current data retained if rows with any null in required columns dropped: {((len(data)-(data[uncompensated_nulls].isnull().any(axis=1).sum()))/len(data))}')
print()
print(data[uncompensated_nulls].isnull().sum().sort_values(ascending=False))

Uncompensated Nulls (total patient revenue)
Current rows: 23449
Rows with any null in required columns: 976
Percent of current data retained if rows with any null in required columns dropped: 0.9583777559810653

Total Unreimbursed and Uncompensated Care    829
Total Patient Revenue                        355
dtype: int64


In [42]:
print(f'Uncompensated Nulls (Total IP & OP Charges)')
print(f"Current rows: {len(data)}")
print(f"Rows with any null in required columns: {data[uncompensated_nulls2].isnull().any(axis=1).sum()}")
print(f'Percent of current data retained if rows with any null in required columns dropped: {((len(data)-(data[uncompensated_nulls2].isnull().any(axis=1).sum()))/len(data))}')
print()
print(data[uncompensated_nulls2].isnull().sum().sort_values(ascending=False))

Uncompensated Nulls (Total IP & OP Charges)
Current rows: 23449
Rows with any null in required columns: 829
Percent of current data retained if rows with any null in required columns dropped: 0.9646466800289991

Total Unreimbursed and Uncompensated Care        829
Combined Outpatient + Inpatient Total Charges      4
dtype: int64


In [43]:
# Dropping null values from columns with very low null rates 
data = data.dropna(subset=['Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds', 'Hospital Total Bed Days Available For Adults & Peds', 'Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds', 'Hospital Total Days Title XVIII For Adults & Peds'])
data.shape

(23148, 118)

In [44]:
print(f'Medicaid Nulls (medicaid_reported)')
print(f"Current rows: {len(data)}")
print(f"Rows with any null in required columns: {data[medicaid_nulls].isnull().any(axis=1).sum()}")
print(f'Percent of current data retained if rows with any null in required columns dropped: {((len(data)-(data[medicaid_nulls].isnull().any(axis=1).sum()))/len(data))}')
print()
print(data[medicaid_nulls].isnull().sum().sort_values(ascending=False))

Medicaid Nulls (medicaid_reported)
Current rows: 23148
Rows with any null in required columns: 2386
Percent of current data retained if rows with any null in required columns dropped: 0.896924140314498

Hospital Total Days Title XIX For Adults & Peds                      2386
Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds       0
dtype: int64


In [45]:
print(f'Uncompensated Nulls (total patient revenue)')
print(f"Current rows: {len(data)}")
print(f"Rows with any null in required columns: {data[uncompensated_nulls].isnull().any(axis=1).sum()}")
print(f'Percent of current data retained if rows with any null in required columns dropped: {((len(data)-(data[uncompensated_nulls].isnull().any(axis=1).sum()))/len(data))}')
print()
print(data[uncompensated_nulls].isnull().sum().sort_values(ascending=False))

Uncompensated Nulls (total patient revenue)
Current rows: 23148
Rows with any null in required columns: 873
Percent of current data retained if rows with any null in required columns dropped: 0.9622861586314152

Total Unreimbursed and Uncompensated Care    726
Total Patient Revenue                        329
dtype: int64


In [46]:
print(f'Uncompensated Nulls (Total IP & OP Charges)')
print(f"Current rows: {len(data)}")
print(f"Rows with any null in required columns: {data[uncompensated_nulls2].isnull().any(axis=1).sum()}")
print(f'Percent of current data retained if rows with any null in required columns dropped: {((len(data)-(data[uncompensated_nulls2].isnull().any(axis=1).sum()))/len(data))}')
print()
print(data[uncompensated_nulls2].isnull().sum().sort_values(ascending=False))

Uncompensated Nulls (Total IP & OP Charges)
Current rows: 23148
Rows with any null in required columns: 726
Percent of current data retained if rows with any null in required columns dropped: 0.9686365992742354

Total Unreimbursed and Uncompensated Care        726
Combined Outpatient + Inpatient Total Charges      0
dtype: int64


## Feature Engineering

In [47]:
# Create indicator for missing values in features
data['medicaid_reported'] = data['Hospital Total Days Title XIX For Adults & Peds'].notnull().astype(int)
data['uncompensated_reported'] = (data['Total Unreimbursed and Uncompensated Care'].notnull() & data['Total Patient Revenue'].notnull()).astype(int)

# Fill nulls with 0 for ratio calculation
data['Hospital Total Days Title XIX For Adults & Peds'] = data['Hospital Total Days Title XIX For Adults & Peds'].fillna(0)
data['Total Unreimbursed and Uncompensated Care'] = data['Total Unreimbursed and Uncompensated Care'].fillna(0)
data['Total Patient Revenue'] = data['Total Patient Revenue'].fillna(np.nan)

# Create features to normalize for hospital size
data['medicare_pct'] = (data['Hospital Total Days Title XVIII For Adults & Peds'] / 
                        data['Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds'])

data['medicaid_pct'] = (data['Hospital Total Days Title XIX For Adults & Peds'] /
                        data['Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds'])

data['bed_utilization'] = (data['Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds'] /
                           data['Hospital Total Bed Days Available For Adults & Peds'])

data['avg_length_of_stay'] = (data['Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds'] /
                              data['Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds'])

data['uncompensated_care_pct'] = (data['Total Unreimbursed and Uncompensated Care'] /
                                  data['Total Patient Revenue'])

# Group Control Types into logical groups
conditions = [
    data['Type of Control'].between(1, 2),
    data['Type of Control'].between(3, 6),
    data['Type of Control'].between(7, 13)
]

choices = ['Non-Profit', 'For-Profit', 'Government']

data['control_type'] = np.select(conditions, choices, 'ERROR')

# Target variable
data['financial_distress'] = (data['Net Income'] < 0).astype(int)

print(f'Shape after feature engineering: {data.shape}')

Shape after feature engineering: (23148, 127)


In [48]:
new_cols = [
    'medicare_pct', 'medicaid_pct', 'bed_utilization',
    'avg_length_of_stay', 'uncompensated_care_pct', 
    'medicaid_reported', 'uncompensated_reported',
    'control_type', 'financial_distress'
]

data[new_cols].describe(include='all')

,medicare_pct,medicaid_pct,bed_utilization,avg_length_of_stay,uncompensated_care_pct,medicaid_reported,uncompensated_reported,control_type,financial_distress
count,23148.000000,23148.000000,23148.000000,23148.000000,2.281900e+04,23148.000000,23148.000000,23148,23148.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Non-Profit,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13966,NaN
mean,0.373706,0.075668,0.396941,3.573917,5.399709e+02,0.896924,0.962286,NaN,0.315578
std,0.175323,0.099457,0.260314,2.548325,4.716730e+04,0.304065,0.190507,NaN,0.464756
min,0.000054,0.000000,0.000011,0.006922,-2.529337e-06,0.000000,0.000000,NaN,0.000000
25%,0.254578,0.014300,0.159250,2.841805,9.829191e-03,1.000000,1.000000,NaN,0.000000
50%,0.351397,0.040258,0.381318,3.398345,1.831975e-02,1.000000,1.000000,NaN,0.000000
75%,0.471276,0.097881,0.610526,4.003053,3.320784e-02,1.000000,1.000000,NaN,1.000000


In [49]:
print(f'Hospitals with medicare_pct > 0.5: {(data['medicare_pct'] > 0.5).sum()}')
print(f'Hospitals with medicare_pct > 0.75: {(data['medicare_pct'] > 0.75).sum()}')
print(f'Hospitals with medicare_pct = 1.0: {(data['medicare_pct'] == 1.0).sum()}')
print()
print(f'Hospitals with medicaid_pct > 0.25: {(data['medicaid_pct'] > 0.25).sum()}')
print(f'Hospitals with medicaid_pct > 0.5: {(data['medicaid_pct'] > 0.5).sum()}')
print(f'Hospitals with medicaid_pct > 0.75: {(data['medicaid_pct'] > 0.75).sum()}')
print(f'Hospitals with medicaid_pct = 1.0: {(data['medicaid_pct'] == 1.0).sum()}')
print()
print(f'Hospitals with bed_utilization > 0.75: {(data['bed_utilization'] > 0.75).sum()}')
print(f'Hospitals with bed_utilization > 1.0: {(data['bed_utilization'] > 1.0).sum()}')
print(f'Hospitals with bed_utilization = 2.0: {(data['bed_utilization'] > 2.0).sum()}')

Hospitals with medicare_pct > 0.5: 4826
Hospitals with medicare_pct > 0.75: 802
Hospitals with medicare_pct = 1.0: 38

Hospitals with medicaid_pct > 0.25: 1377
Hospitals with medicaid_pct > 0.5: 219
Hospitals with medicaid_pct > 0.75: 21
Hospitals with medicaid_pct = 1.0: 0

Hospitals with bed_utilization > 0.75: 2332
Hospitals with bed_utilization > 1.0: 63
Hospitals with bed_utilization = 2.0: 1


In [50]:
data[data['bed_utilization'] > 1.0][['Hospital Name', 'Year', 'Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds', 'Hospital Total Bed Days Available For Adults & Peds', 'bed_utilization']].sort_values('bed_utilization', ascending=False).head(10)

,Hospital Name,Year,Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds,Hospital Total Bed Days Available For Adults & Peds,bed_utilization
22206,MEMORIAL HERMANN MEMORIAL CITY MEDIC,2022,63844.0,13140.0,4.858752
22694,IZARD REGIONAL HOSPITAL,2022,163.0,107.0,1.523364
19559,ADVENTHEALTH PALM COAST,2022,43608.0,29565.0,1.474987
15588,LLUMC MURRIETA,2021,47588.0,33945.0,1.401915
23682,NORTHSIDE HOSPITAL - DULUTH,2022,29527.0,21170.0,1.394757
21882,LLUMC MURRIETA,2022,46066.0,33945.0,1.357078
13996,ADVENTHEALTH PALM COAST,2021,39091.0,29565.0,1.322205
25574,ADVENTHEALTH PALM COAST,2023,38732.0,29565.0,1.310063
9380,LONG ISLAND JEWISH MEDICAL CENTER,2020,283345.0,225332.0,1.257456
16215,HARBORVIEW MEDICAL CENTER,2021,104087.0,84680.0,1.229180


In [51]:
data[data['Hospital Name'].str.contains('MEMORIAL HERMANN MEMORIAL CITY')][['Hospital Name', 'Year', 'Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds', 'Hospital Total Bed Days Available For Adults & Peds', 'bed_utilization']]

,Hospital Name,Year,Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds,Hospital Total Bed Days Available For Adults & Peds,bed_utilization
6032,MEMORIAL HERMANN MEMORIAL CITY MEDIC,2019,60882.0,131760.0,0.462067
7264,MEMORIAL HERMANN MEMORIAL CITY MEDIC,2020,63894.0,131760.0,0.484927
15519,MEMORIAL HERMANN MEMORIAL CITY MEDIC,2021,61033.0,131400.0,0.464482
22206,MEMORIAL HERMANN MEMORIAL CITY MEDIC,2022,63844.0,13140.0,4.858752
27989,MEMORIAL HERMANN MEMORIAL CITY MEDIC,2023,65689.0,131400.0,0.499916


In [55]:
data[data['medicare_pct'] == 1.0][['Hospital Name', 'Year', 'CCN Facility Type', 'Hospital Total Days Title XVIII For Adults & Peds', 'Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds']].drop_duplicates('Hospital Name')

,Hospital Name,Year,CCN Facility Type,Hospital Total Days Title XVIII For Adults & Peds,Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds
1435,NIOBRARA VALLEY HOSPITAL,2019,CAH,20.0,20.0
1988,GARFIELD CO. HEALTH CENTER,2019,CAH,18.0,18.0
2061,KENMARE COMMUNITY HOSPITAL,2019,CAH,10.0,10.0
2181,PHILLIPS EYE INSTITUTE,2019,STH,4.0,4.0
2409,HALE HOOLA HAMAKUA,2019,CAH,8.0,8.0
2455,CATALINA ISLAND MEDICAL CENTER,2019,CAH,20.0,20.0
2484,SAMUEL MAHELONA MEMORIAL HOSPITAL,2019,CAH,4.0,4.0
6387,LUMBERTON HOSPITAL LLC,2020,STH,320.0,320.0
6599,WARREN MEMORIAL HOSPITA,2020,CAH,20.0,20.0
6738,CAH # 16 - HASKELL,2020,CAH,2.0,2.0


In [57]:
data = data[~data['Hospital Name'].str.contains('BALTIMORE CONVENTION')]

In [58]:
data[data['avg_length_of_stay'] > 30][['Hospital Name', 'Year', 'avg_length_of_stay', 'CCN Facility Type']].sort_values('avg_length_of_stay', ascending=False).head(20)

,Hospital Name,Year,avg_length_of_stay,CCN Facility Type
8677,TEXAS CENTER FOR INFECTIOUS DISEASE,2020,153.462687,STH
17174,TEXAS CENTER FOR INFECTIOUS DISEASE,2021,148.343284,STH
1138,TEXAS CENTER FOR INFECTIOUS DISEASE,2019,139.810127,STH
28828,HIRAM W DAVIS MEDICAL CENTER - MS,2023,98.555556,STH
27189,ALTUS HOUSTON HOSPITAL LP,2023,85.600000,STH
8489,HEALTHBRIDGE CHILDRENS HOSPITAL,2020,83.713235,CH
17311,HIRAM W DAVIS MEDICAL CENTER - MS,2021,73.375000,STH
22369,RANKEN JORDAN,2022,68.596078,CH
6942,HIRAM W DAVIS MEDICAL CENTER - MS,2020,58.000000,STH
1475,RANKEN JORDAN,2019,54.630282,CH


In [59]:
exclude_ltach = [
    'TEXAS CENTER FOR INFECTIOUS DISEASE',
    'HIRAM W DAVIS MEDICAL CENTER - MS',
    'HACKENSACK MERIDIAN LTACH',
    'LEVINDAL HEBREW GER. CTR. & HOSPT.'
]
data = data[~data['Hospital Name'].isin(exclude_ltach)]
print(f'Shape after removing non-acute long-stay facilities: {data.shape}')

Shape after removing non-acute long-stay facilities: (23132, 127)


In [60]:
data[data['Hospital Name'].str.contains('ALTUS HOUSTON')][['Hospital Name', 'Year', 'avg_length_of_stay', 'CCN Facility Type']]

,Hospital Name,Year,avg_length_of_stay,CCN Facility Type
7080,ALTUS HOUSTON HOSPITAL LP,2020,1.000000,STH
15277,ALTUS HOUSTON HOSPITAL LP,2021,1.000000,STH
21231,ALTUS HOUSTON HOSPITAL LP,2022,0.471104,STH
27189,ALTUS HOUSTON HOSPITAL LP,2023,85.600000,STH


In [61]:
data[data['Hospital Name'] == 'ALTUS HOUSTON HOSPITAL LP'][['Year', 'Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds', 'Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds', 'avg_length_of_stay']]

,Year,Hospital Total Days (V + XVIII + XIX + Unknown) For Adults & Peds,Hospital Total Discharges (V + XVIII + XIX + Unknown) For Adults & Peds,avg_length_of_stay
7080,2020,1756.0,1756.0,1.000000
15277,2021,1756.0,1756.0,1.000000
21231,2022,2739.0,5814.0,0.471104
27189,2023,428.0,5.0,85.600000


In [62]:
data = data[data['Hospital Name'] != 'ALTUS HOUSTON HOSPITAL LP']
print(f'Shape after removing Altus Houston: {data.shape}')
print(f"Rows with avg_length_of_stay > 30: {(data['avg_length_of_stay'] > 30).sum()}")

Shape after removing Altus Houston: (23128, 127)
Rows with avg_length_of_stay > 30: 10


In [63]:
data[data['avg_length_of_stay'] > 30][['Hospital Name', 'Year', 'avg_length_of_stay', 'CCN Facility Type']].sort_values('avg_length_of_stay', ascending=False)

,Hospital Name,Year,avg_length_of_stay,CCN Facility Type
8489,HEALTHBRIDGE CHILDRENS HOSPITAL,2020,83.713235,CH
22369,RANKEN JORDAN,2022,68.596078,CH
1475,RANKEN JORDAN,2019,54.630282,CH
30322,FRANCISCAN HOSPITAL FOR CHILDREN,2023,45.829268,CH
2323,KENNEDY KRIEGER,2019,43.188854,CH
13298,FRANCISCAN HOSPITAL FOR CHILDREN,2021,40.476103,CH
19307,CHILDRENS SPECIALIZED HOPSITAL,2022,38.043810,CH
13647,CHILDRENS SPECIALIZED HOPSITAL,2021,35.014286,CH
26032,INOVA SPECIALTY HOSPITAL,2023,33.951613,STH
856,PORTERVILLE DEVELOPMENTAL CENTER,2019,30.833333,STH


In [64]:
data[data['Hospital Name'].isin(['INOVA SPECIALTY HOSPITAL', 'PORTERVILLE DEVELOPMENTAL CENTER'])][['Hospital Name', 'Year', 'avg_length_of_stay', 'CCN Facility Type']]

,Hospital Name,Year,avg_length_of_stay,CCN Facility Type
856,PORTERVILLE DEVELOPMENTAL CENTER,2019,30.833333,STH
6741,PORTERVILLE DEVELOPMENTAL CENTER,2020,28.666667,STH
24177,PORTERVILLE DEVELOPMENTAL CENTER,2022,14.000000,STH
26032,INOVA SPECIALTY HOSPITAL,2023,33.951613,STH


In [65]:
data = data[data['Hospital Name'] != 'PORTERVILLE DEVELOPMENTAL CENTER']
print(f'Shape after removing Porterville: {data.shape}')
print(f"Remaining rows with avg_length_of_stay > 30: {(data['avg_length_of_stay'] > 30).sum()}")

Shape after removing Porterville: (23125, 127)
Remaining rows with avg_length_of_stay > 30: 9
